# TPC distortion corrections from the CM laser: current performance and the updated model
*(meeting version; every figure: two panels, South | North, unless noted)*

**Part A:** the current performance, the views from before.

**Part B:** the same studies redone with what we learned. Every figure compares:
- **no correction** (grey);
- **current CDB** (red): the production 3D space-charge maps (α 1.8, gain on, flat φ) at the CDB keff × the block's
  current, with the current static model;
- **best** (black): the new static model + the α 1.6 map at keff = 0.60 × I_side.

**Part C:** the static study, main parts only.

**What changed in the static model:**

| | current CDB | new |
|---|---|---|
| module frames | charge scale −180 | dropped (0) |
| B_r | × 1.0 | × 0.8 |
| IFC vs automatic | 0 | South −60 V, North −120 V |
| field-off reference | fixed, old static model | 78242 corrected with the same model |

Geometry is the same in both: z −40 mm, tilt 0.16 / 0.85 mrad, survey x/y, P/T-corrected CM field.

# Part A: current performance

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
import bsnb as b, bsplots as bp, cmplots as c, v2plots as v, scplots as sp
plt.rcParams['figure.dpi'] = 80
plt.rcParams.update({'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 17, 'legend.fontsize': 13,
                     'xtick.labelsize': 14, 'ytick.labelsize': 14, 'figure.titlesize': 17})
b.plt.rcParams.update(plt.rcParams)
CFG = {'raw': 'no correction',
       'static': 'static correction only (z -40 mm, tilt 0.16/0.85 mrad, P/T CM field, IFC automatic), keff 0',
       'sc16': 'alpha 1.6 map, keff = 0.596 x I_side (79513/0: S 1.281, N 1.347); high lumi b 0.685 / 0.652; + static as above',
       'sc20': 'alpha 2.0 map, keff = 0.665 x I_side (79513/0: S 1.429, N 1.503); + static',
       'cdbmap': 'default CDB map, CDB keff x current (79513/0: S 1.589, N 1.590); + static',
       'cdbk': 'alpha 2.0 map, CDB keff x current (79513/0: S 1.589, N 1.590); + static',
       'final': 'alpha 2.0 map, laser-absolute keff = 0.843 x I_side (79513/0: S 1.811, N 1.904); + static'}
def cfg(*stages, extra=''):
    print('CONFIG  ' + ('\n        '.join(f"{st:7s}: {CFG[st]}" for st in stages)) + (f"\n        {extra}" if extra else ''))
def show(*figs):
    for f in figs:
        display(f)
    plt.close('all')
R = c.RC[c.RING]

## 1. The central membrane and its laser image
**What is plotted.**
- Left: the CM stripe positions (the "truth" pattern) in x–y for one side.
- Right: a zoom on one sector, with the reconstructed laser clusters (run 79513, block 0) on top, before any
  correction and after the full correction.

**Why.** Each laser cluster sits on a known stripe. The shift between cluster and stripe is the distortion we correct.
Raw shifts are millimetres to centimetres.

In [ ]:
low, high = v.blocks_low_high()
def truth_xy(stage, blk='79513_00000'):
    a = c.load(f'T_{stage}', blk)
    tR = a[:, 1] - a[:, 3] / 10.0                 # truth = reco - residual (dR mm -> cm)
    tphi = a[:, 2] - a[:, 4] / 10.0 / a[:, 1]     # R dphi mm -> rad
    return a, tR, tphi
fig, axs = plt.subplots(1, 2, figsize=(16, 7.5))
cfg('raw', 'sc16', extra='run 79513 block 0, South')
for stage, col, lab in (('raw', 'tab:red', 'reconstructed, no correction'), ('sc16', 'tab:green', 'after the full correction')):
    a, tR, tphi = truth_xy(stage)
    s = a[:, 0] == 0
    if stage == 'raw':
        axs[0].plot(tR[s] * np.cos(tphi[s]), tR[s] * np.sin(tphi[s]), '.', ms=2, color='k', label='CM stripes (truth)')
        axs[1].plot(tR[s] * np.cos(tphi[s]), tR[s] * np.sin(tphi[s]), 'o', ms=6, mfc='none', color='k', label='CM stripes (truth)')
    axs[1].plot(a[s, 1] * np.cos(a[s, 2]), a[s, 1] * np.sin(a[s, 2]), '.', ms=6, color=col, label=lab)
axs[0].set_aspect('equal'); axs[0].set_xlabel('x [cm]'); axs[0].set_ylabel('y [cm]'); axs[0].set_title('South: CM stripe pattern (laser-matched stripes)')
axs[1].set_xlim(30, 75); axs[1].set_ylim(-5, 22); axs[1].set_xlabel('x [cm]'); axs[1].set_ylabel('y [cm]')
axs[1].set_title('zoom: stripes (open) and laser clusters'); axs[1].legend(loc='upper left')
fig.tight_layout(); show(fig)

## 2. Ben-style view (matcher QA): every matched cluster
**What is plotted.** Each point is one laser cluster matched to its CM stripe: dR (left) and Δφ (right, rad) against R,
coloured by φ bin, as in the LaserClusterTruthMatcher QA. Uncorrected and fully corrected, for each side (79513 block 0).

In [ ]:
def ben_scatter(stage, side):
    a = c.load(f'T_{stage}', '79513_00000'); a = a[a[:, 0] == side]
    sec = np.floor((a[:, 2] % (2 * np.pi)) / (np.pi / 6)).astype(int)
    fig, axs = b.pair(f"2) {c.SIDES[side]}, {b.STG[stage][0]}: matched clusters vs CM truth (79513/0)")
    for k, col in zip(range(12), plt.cm.hsv(np.linspace(0, 0.92, 12))):
        sel = sec == k
        axs[0].plot(a[sel, 1], a[sel, 3] / 10, '.', ms=4, color=col, label=f"phi bin {k}" if k % 2 == 0 else None)
        axs[1].plot(a[sel, 1], a[sel, 4] / 10 / a[sel, 1], '.', ms=4, color=col)
    axs[0].set_ylabel('dR [cm]'); axs[1].set_ylabel('dphi [rad]')
    for ax in axs:
        ax.set_xlabel('R [cm]'); ax.axhline(0, color='k', lw=0.8)
    axs[0].legend(fontsize=11, markerscale=3, ncol=2); fig.tight_layout()
    return fig
for side in (0, 1):
    cfg('raw', 'sc16', extra=f'run 79513 block 0, {c.SIDES[side]}')
    show(ben_scatter('raw', side))

**The same at the production (CDB) keff.** keff = the run's CDB keff × the segment current correction (79513/0: South
1.589, North 1.590), with the α 2.0 map and with the default CDB map.

In [ ]:
for side in (0, 1):
    cfg('cdbk', 'cdbmap', extra=f'run 79513 block 0, {c.SIDES[side]}')
    show(ben_scatter('cdbk', side))

### 2c. Ben's matcher QA graphs (LaserClusterTruthMatcher output, 79513 block 0)
**What is plotted.** The matcher's own TGraphs: ΔR and Δφ vs R_reco in 38 φ bins (all bins overlaid, colour = φ bin)
for South (negz) and North (posz). Shown for: no correction, static correction only, and the α 1.6 map at a = 0.39 ×
I_R1 (keff South 1.344 / North 1.282, close to the space-charge keff 1.281 / 1.347). Then the matcher's reconstructed vs
truth points, and its integrated distortion maps.

In [ ]:
import ROOT
ROOT.gROOT.SetBatch(True)
TALK = 'data/talk'
FILES = (('noGarfield', 'no correction', 'no PHGarfield correction'),
         ('st_nom', 'static only', 'static correction only (z -40 mm, tilt 0.16/0.85, P/T, IFC automatic), keff 0'),
         ('cdbmap', 'default CDB map, CDB keff', 'default CDB map, keff = CDB x current = S 1.589 / N 1.590; + static'))
def graphs(tag, side, what):
    f = ROOT.TFile.Open(f'{TALK}/79513_0_{tag}.root'); z = 'negz' if side == 0 else 'posz'
    out = []
    for i in range(38):
        g = f.Get(f'{what}_phiBin{i}_{z}')
        if g and g.GetN():
            out.append((i, np.array([g.GetX()[j] for j in range(g.GetN())]), np.array([g.GetY()[j] for j in range(g.GetN())])))
    f.Close()
    return out
def ben_graphs(tag, lab):
    figs = []
    for side in (0, 1):
        fig, axs = b.pair(f"2c) matcher QA, {c.SIDES[side]}, {lab}: dR and dphi vs R in 38 phi bins")
        for ax, what, yl in ((axs[0], 'dR', 'delta R [cm]'), (axs[1], 'dPhi', 'delta phi [rad]')):
            for i, x, y in graphs(tag, side, what):
                ax.plot(x, y, '-o', ms=3, lw=1, color=plt.cm.hsv(i / 38 * 0.92))
            ax.set_xlabel('R_reco [cm]'); ax.set_ylabel(yl); ax.axhline(0, color='k', lw=0.8)
        fig.tight_layout(); figs.append(fig)
    return figs
for tag, lab, conf in FILES:
    print('CONFIG  run 79513 block 0, ' + conf)
    show(*ben_graphs(tag, lab))
# reco vs truth points and the integrated distortion maps
for tag, lab, conf in (FILES[0], FILES[2]):
    print('CONFIG  run 79513 block 0, ' + conf)
    f = ROOT.TFile.Open(f'{TALK}/79513_0_{tag}.root')
    # the matcher stores the points as (phi [rad], R [cm]): x-y view and a zoomed R-phi view
    #fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    #fig.suptitle(f"2c) matcher QA, {lab}: reconstructed (colour) and truth (black) CM points, x-y", fontsize=17)
    #fig2, axs2 = b.pair(f"2c) matcher QA, {lab}: the same, R vs phi (zoom 0-60 deg)")
    #for side, z in ((0, 'negz'), (1, 'posz')):
    #    for nm, col, ms in ((f'gr_truthPoints_{z}', 'k', 3), (f'gr_recoPoints_{z}', 'tab:red' if tag == 'noGarfield' else 'tab:green', 2)):
    #        g = f.Get(nm)
    #        if g and g.GetN():
    #            ph = np.array([g.GetX()[j] for j in range(g.GetN())]); rr = np.array([g.GetY()[j] for j in range(g.GetN())])
    #            lab3 = 'truth (CM stripes)' if 'truth' in nm else 'reconstructed'
    #            axs[side].plot(rr * np.cos(ph), rr * np.sin(ph), '.', ms=ms, color=col, label=lab3)
    #            sel = (ph >= 0) & (ph < np.pi / 3)
    #            axs2[side].plot(np.degrees(ph[sel]), rr[sel], '.', ms=ms + 1, color=col, label=lab3)
    #    axs[side].set_title(c.SIDES[side]); axs[side].set_aspect('equal'); axs[side].set_xlabel('x [cm]'); axs[side].set_ylabel('y [cm]')
    #    axs2[side].set_title(c.SIDES[side]); axs2[side].set_xlabel('phi [deg]'); axs2[side].set_ylabel('R [cm]')
    #axs[1].legend(markerscale=5, loc='upper left', bbox_to_anchor=(1.0, 1.0))
    #axs2[1].legend(markerscale=4, loc='upper left', bbox_to_anchor=(1.0, 1.0))
    #fig.tight_layout(); fig2.tight_layout(); show(fig, fig2)
    fig, axs = b.pair(f"2c) matcher QA, {lab}: integrated distortion maps (hIntDistortion, South)")
    for ax, nm, lab2 in ((axs[0], 'hIntDistortionR_negz', 'delta R'), (axs[1], 'hIntDistortionP_negz', 'delta phi')):
        h = f.Get(nm)
        if h:
            nx, ny = h.GetNbinsX(), h.GetNbinsY()
            A = np.array([[h.GetBinContent(i, j) for i in range(1, nx + 1)] for j in range(1, ny + 1)])
            xe = [h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)]; ye = [h.GetYaxis().GetBinLowEdge(j) for j in range(1, ny + 2)]
            vmax = np.nanpercentile(np.abs(A[A != 0]), 95) if (A != 0).any() else 1
            im = ax.pcolormesh(xe, ye, A, cmap='coolwarm', vmin=-vmax, vmax=vmax); fig.colorbar(im, ax=ax, label=lab2)
            ax.set_xlabel(h.GetXaxis().GetTitle() or 'x'); ax.set_ylabel(h.GetYaxis().GetTitle() or 'y'); ax.set_title(nm)
    fig.tight_layout(); show(fig)
    f.Close()

### 2b. The same as medians: residual vs R in φ slices
**What is plotted.** Cluster minus matched stripe, RΔφ (top) and dR (bottom), against R. Colour = φ slice of 30°.
Raw on the left, after the full correction on the right (South, 79513/0).

**Why.** This is the classic CM-laser picture. Before correction RΔφ reaches −10 … −20 mm and depends on R and φ; after
correction everything collapses to around 0. The remaining structure is what the rest of the talk is about.

**Note on raw dR (+12 … +25 mm).** Without any correction, the clusters are so far off that the nearest stripe is
often the neighbouring stripe row (row spacing 11–22 mm). The raw dR vs truth is therefore partly a wrong-row offset.
After the static correction, the clusters land on the right rows (section 3, matching QA). That is why the matching
is done after the correction.

In [ ]:
def ben(stage, side, q, ax, title):
    a = c.load(f'T_{stage}', '79513_00000'); a = a[a[:, 0] == side]
    sec = np.floor((a[:, 2] % (2 * np.pi)) / (np.pi / 6)).astype(int)
    for k, col in zip(range(12), plt.cm.hsv(np.linspace(0, 0.92, 12))):
        m = c.ring_stats(a[sec == k], side, q)[0]
        ax.plot(R, m, '-o', ms=3, color=col, lw=1.2, label=f"{30*k}-{30*k+30}" if k % 2 == 0 else None)
    ax.set_title(title); ax.set_xlabel('R [cm]'); ax.set_ylabel(b.QL[q]); ax.axhline(0, color='k', lw=0.8)
cfg('raw', 'cdbk', extra='run 79513 block 0, vs CM truth')
for q in (4, 3):
    for side in (0, 1):
        fig, axs = b.pair(f"2) {b.QL[q].split(' [')[0]} vs R per 30 deg slice, {c.SIDES[side]}, 79513/0, vs CM truth")
        ben('raw', side, q, axs[0], 'no correction'); ben('cdbk', side, q, axs[1], 'full correction (alpha 2.0, CDB keff)')
        axs[0].legend(fontsize=9, title='phi [deg]'); fig.tight_layout(); show(fig)

# Part B: updated performance
Stages: **no correction**, **current CDB**, **best** (new static + α 1.6 at b 0.60). The field-off reference
is corrected with the same static model as the beam run it is compared with; beam − no beam also uses the same model on both
sides.

In [ ]:
import nsqa as n, staticqa as sq
BEST, B_MAP = n.BEST, n.BESTMAP

## B1. Matched clusters vs CM truth (79513 block 0)
**What is plotted.** Each point is one laser cluster matched to its CM stripe: dR (left) and Δφ (right) against R, coloured
by φ bin. Current CDB, then best, for each side.

In [ ]:
for side in (0, 1):
    n.cfg('cdb', BEST, extra=f'run 79513 block 0, {c.SIDES[side]}')
    show(n.scatter_fig('cdb', side), n.scatter_fig(BEST, side))

## B2. Residual vs R in 30° φ slices
**What is plotted.** Median RΔφ (and dR) per R ring, one curve per 30° slice:
- left = current CDB, right = best;
- vs CM truth (79513/0), then beam − no beam 75073 (795xx pooled).

In [ ]:
n.cfg('raw', 'cdb', BEST, extra='run 79513 block 0, vs CM truth')
for q in (4, 3):
    for side in (0, 1):
        show(n.slices_fig('raw', 'cdb', side, q, what='(79513/0)'), n.slices_fig('cdb', BEST, side, q, what='(79513/0)'))
n.cfg('raw', 'cdb', BEST, extra='beam - no beam 75073, all 11 low-lumi blocks pooled')
for side in (0, 1):
    show(n.slices_fig('raw', 'cdb', side, 4, cmp='NB75', blks=low, what='(795xx)'), n.slices_fig('cdb', BEST, side, 4, cmp='NB75', blks=low, what='(795xx)'))

## B3. What has to be corrected and what is left: median per R ring
**What is plotted.** Median per R ring, 11 low-lumi blocks pooled. Each comparison is shown twice: no correction vs current
CDB, then current CDB vs best. Comparisons:
- vs CM truth;
- beam − no beam 75073;
- beam − field off (same static model).

In [ ]:
n.cfg('raw', 'cdb', BEST, extra='11 low-lumi blocks pooled (795xx)')
for cmp in ('T', 'NB75', 'OFF'):
    for q in (5, 3):
        show(b.b_vs_r(cmp, q, names=('raw', 'cdb')), b.b_vs_r(cmp, q, names=('cdb', BEST)))

**Note on CM truth.** Field off vs truth (no B at all) already shows a constant dφ of about −2 (South) and −3 (North) mrad
and a dR of about +5 mm: the CM stripes are rotated and scaled against the pads. A constant dφ is a rigid rotation, which
the TPC alignment absorbs. The field-off and no-beam references remove it, so they are the main references here.

## B4. Space charge alone: beam − no beam
**What is plotted.**
- The median per ring for the high-luminosity runs (80892, 81558).
- The 2D residual map (φ, R) for current CDB and best.

In [ ]:
n.cfg('cdb', BEST, extra='beam - no beam 75073')
show(b.b_vs_r('NB75', 5, names=('raw', 'cdb'), lumi='high'), b.b_vs_r('NB75', 5, names=('cdb', BEST), lumi='high'))
show(b.b_map('NB75', 'cdb', q=4), b.b_map('NB75', BEST, q=4))

### B4a. Which map, which keff (new static)
**What is plotted.** For four maps, against b (keff = b × I_side), 795xx and high luminosity:
- the flatness (rms of the ring dφ about its mean);
- the sagitta (dφ slope).

The legend gives the best b.

In [ ]:
print('CONFIG  new static (frame 0, B_r 0.8, IFC S -60 / N -120 V) + map x b, beam - no beam 75073, keff = b x I_side')
show(n.scan_fig('NB75', 'flat'), n.scan_fig('NB75', 'sag'))
show(n.scan_fig('NB75', 'flat', lumi='high'), n.scan_fig('NB75', 'sag', lumi='high'))

**Take-away.**
- α 1.6 is the flattest map, at b ≈ 0.62 (South) and 0.59–0.60 (North).
- The same b at high luminosity (0.63 / 0.59): keff stays linear in the GEM current.
- α 2.0 is never as flat and needs a higher b.

### B4b. keff is linear in the GEM current
**What is plotted.** The flattest keff per block (α 1.6, beam − no beam) against the side's GEM current. A line through 0
means keff = b × I_side.

In [ ]:
show(n.keff_current_fig('NB75'))

### B4c. What is left: φ structure
**What is plotted.**
- RΔφ vs φ for the three module rings (R1, R2, R3), current CDB and best.
- The sagitta per 30° sector.
- RΔφ per sector after removing that sector's rotation.

In [ ]:
n.cfg('cdb', BEST, extra='beam - no beam 75073, 795xx pooled')
show(b.b_vs_phi('NB75', 4, st='cdb'), b.b_vs_phi('NB75', 4, st=BEST))
show(n.sector_sag_fig('NB75', ('cdb', BEST)), n.sector_sag_fig('OFF', ('cdb', BEST)))
show(n.rot_removed('NB75', 'cdb'), n.rot_removed('NB75', BEST))

## B5. The absolute keff now agrees
With the old static model, beam − no beam wanted b ≈ 0.60, but the absolute references (field off, truth) wanted
0.79–0.84. The extra keff was making up for the static distortion we under-estimated. With the new static model, every
reference agrees.

**What is plotted.**
- The median per ring vs field off and vs CM truth.
- The b each reference / criterion prefers for α 1.6 and α 2.0. The dashed line is the track rDCA result with the **old**
  static model.

In [ ]:
n.cfg('cdb', BEST)
show(b.b_vs_r('OFF', 5, names=('cdb', BEST)), b.b_vs_r('T', 5, names=('cdb', BEST)))
show(n.scan_fig('OFF', 'sag'), n.scan_fig('OFF', 'flat'))
show(n.votes_fig(B_MAP), n.votes_fig('prod2p0'))

**Take-away.**
- α 1.6, new static: b = 0.59–0.64 (South) and 0.55–0.60 (North) from beam − no beam 75073, beam − field off and the
  flatness vs truth. The space-charge and absolute keff agree to a few percent.
- 83319 sits apart: North low, South high, as in every study.
- The tracks (rDCA) have to be redone with the new static model. With the old one they wanted 0.73 / 0.77.

# Part C: the static model
## C1. Every comparison, uncorrected and corrected
**What is plotted.** dφ and dR vs R (and vs φ) for:
- field off vs truth;
- no beam vs truth;
- no beam − field off;
- beam vs truth;
- beam − field off;
- beam − no beam.

Three versions: no correction, current CDB, new static (+ best space charge for beam).

In [ ]:
print('CONFIG  no correction')
show(sq.overview('raw', 'raw', 'dphi', 'R', 'no correction'), sq.overview('raw', 'raw', 'dR', 'R', 'no correction'))
n.cfg('cdbst', 'cdb')
show(sq.overview('cdb', 'cdbst', 'dphi', 'R', 'current CDB'), sq.overview('cdb', 'cdbst', 'dR', 'R', 'current CDB'))
n.cfg('nsst', BEST)
show(sq.overview(BEST, 'nsst', 'dphi', 'R', 'new static + best'), sq.overview(BEST, 'nsst', 'dR', 'R', 'new static + best'))
show(sq.overview(BEST, 'nsst', 'dphi', 'phi', 'new static + best'), sq.overview(BEST, 'nsst', 'dR', 'phi', 'new static + best'))

## C2. B_r scale: shifts dφ, cannot change its shape
Near the axis, B_r ≈ −(R/2) ∂B_z/∂z, so B_r ∝ R. The E×B displacement RΔφ ∝ R, and dφ moves by the same amount at every R.

**What is plotted.** No beam − field off for B_r × 0.5 … 1.2:
- dφ vs R;
- the median dφ and dR against the scale.

This is an earlier scan, before frames were dropped and with a fixed field-off reference; the frame does not change the
point.

In [ ]:
print('CONFIG  no beam 75073 - field off, old static (frame -180) + B_r scale')
show(sq.bs_fig('OFF', ['75073'], 'st', 'dphi', 'R', 'no beam 75073 - field off'))
show(sq.bs_summary('dphi'), sq.bs_summary('dR'))

## C3. IFC: the knob for the shape
The inner field-cage voltage sets E_r at small R, so it moves R1 against R2 / R3.

**What is plotted.** Frame 0, field-off reference with the same model, B_r × {0.75, 0.80, 0.85} × IFC lowering per side
(relative to the automatic IFC: South +211 V after the change, North 0):
- the rms of the ring dφ;
- the shape dφ(R1) − dφ(R2,R3);
- dφ vs R at B_r 0.8.

*The B_r × IFC maps and the dφ / dR curves per IFC lowering are in the backup (B-C3).*

**Take-away.**
- The best point is B_r 0.8, IFC South −60 V and North −90 to −120 V: ring-dφ rms 0.3–0.5 mrad, against 3 mrad for
  the old model.
- The South − North IFC difference grows a little (+30 to 60 V), so the 211 V South correction stays.
- Field off vs truth (no B) points the same way for the IFC; the next figure shows it.

In [ ]:
print('CONFIG  field off 78242 vs truth (B = 0), static model with IFC shift; South also with frame 0 (with -180 the South matching fails)')
show(sq.bse_offtruth_fig('dR'))

## C4. TPC tilt (briefly)
**What is plotted.** The first harmonic of RΔφ(φ) (R2 rings) for five TPC rotations, with the new static model: no beam −
field off, and beam − field off (best). A sine that follows the rotation is a tilt.

In [ ]:
print('CONFIG  new static model, TPC rotation varied at z -40; field-off reference with the same model and geometry')
show(n.tilt_fig(['75073'], 'st_ns', 'no beam 75073 - field off'), n.tilt_fig(['79508', '79513', '79516', '79529'], BEST, 'beam 795xx - field off'))

### C4b. Rotation scan around the fit, and x/y shifts
A linear fit of the five rotations gave rot_x ≈ 0.45, rot_y ≈ 1.05 mrad for a flat RΔφ(φ) on both sides. Scan:
- rot_x {0.40, 0.48, 0.56} × rot_y {0.95, 1.05, 1.15} mrad;
- x ±1.5 mm, y +1.5 mm at 0.48 / 1.05 (y +3 mm: the ACTS geometry does not build).

**What is plotted.** The first-harmonic amplitude per module ring, RΔφ and dR, for no beam − field off and beam − field
off.

*The first-harmonic amplitudes for every rotation / shift are in the backup (B-C4b).*

**Take-away.**
- **RΔφ sine:** rot_x ≈ 0.48, rot_y ≈ 1.05 mrad brings it from 0.3–0.5 mm to ≈ 0.1–0.2 mm on both sides, with
  the mean dφ, the ring rms and so the keff unchanged. That lies between the endcap survey (0.30 / 1.49) and the beam
  axis (dy/dz −0.80).
- **dR sine:** it does not follow the rotation or an x/y shift. In South R3 it is 1.7–1.9 mm for 83319 and the 795xx
  beam runs but only 0.2–0.4 mm for 75073. 75073 is the only run before the South IFC change, so this looks like a
  φ-dependent South field-cage feature after the change, not a geometry offset.

## C5. TPC z shift (one slide)

In [ ]:
print('CONFIG  new static model, TPC z varied at tilt 0.16 / 0.85; field-off reference with the same model and geometry')
show(n.z_fig(['75073'], 'st_ns', 'no beam 75073 - field off'), n.z_fig(['79508', '79513', '79516', '79529'], BEST, 'beam 795xx - field off'))

# Where we are: the sagitta per φ sector, every run
**What is plotted.** The sagitta (from the dφ slope over R) per 30° sector for:
- no correction (grey);
- the current CDB (red);
- the new correction (black: new static + α 1.6 at b 0.60).

Low luminosity (795xx) and high luminosity (80892, 81558) separately, for beam − no beam (space charge) and beam − field
off (everything). Then every run on its own, and the rms over the 12 sectors per run.

In [ ]:
n.cfg('raw', 'cdb', BEST)
for cmp in ('NB75', 'OFF'):
    show(n.sag_sector_fig(cmp, 'low'), n.sag_sector_fig(cmp, 'high'))
for cmp in ('NB75', 'OFF'):
    show(n.sag_runs_fig(cmp, 'cdb'), n.sag_runs_fig(cmp, BEST))
show(n.sag_summary_fig('NB75'), n.sag_summary_fig('OFF'))

**What a track sees, per ring.** RΔφ after a straight line per 10° φ bin is removed (a high-pT track absorbs that line):
bias + resolution. Histograms per ring for the current CDB and the new correction (low luminosity, then high), and the
width (MAD) per ring as a summary.

In [ ]:
n.cfg('cdb', BEST)
show(n.track_width_fig('NB75'), n.track_width_fig('OFF'))
show(n.track_width_fig('NB75', 'median'), n.track_width_fig('OFF', 'median'))
for st in ('cdb', BEST):
    show(*c.ring_hists('NB75', st, q=4, mode='line_phi', title=f"beam - no beam 75073, {b.STG[st][0]}: R dphi per ring"))
    show(*c.ring_hists('NB75', st, q=4, mode='line_phi', blks=n.HIGH, title=f"beam - no beam 75073, HIGH lumi, {b.STG[st][0]}: R dphi per ring"))
    show(*c.ring_hists('OFF', st, q=4, mode='line_phi', title=f"beam - field off, {b.STG[st][0]}: R dphi per ring"))

**Take-away.** The track-view width per ring is about the same for the current CDB and the new correction (all rings:
MAD 0.54–0.61 mm at low luminosity, slightly better for the new one at high). It is set by structure inside a 10° bin,
which neither correction models. The gain of the new correction is in the sagitta and the φ structure above.

**Take-away.**
- Beam − field off, per run: the rms of the sector sagitta goes from 0.7–1.7 mm (no correction) to 0.2–0.6 mm (current CDB)
  and 0.14–0.34 mm (new). That's better than the current CDB in every run and on both sides.
- High luminosity, South: a φ structure is left in beam − no beam (sectors at 250–330° low by up to ~0.9 mm). That's the
  module gains, which a single b can't follow. The sector-by-sector gain tuning is the next step there.

# Summary
- **Static model:**
  - module frames dropped;
  - B_r × 0.8 (it only shifts dφ: a rotation-like offset);
  - IFC lowered by about 60 V (South) and 120 V (North) relative to the automatic setting, which fixes the dφ(R) shape.
  - With these, no beam − field off goes from ≈ 3 mrad to ≈ 0.3–0.5 mrad rms.
- **Space charge:** α 1.6 map, keff = b × I_side with b ≈ 0.60–0.62 (South) and 0.55–0.60 (North), the same at high
  luminosity.
- **The space-charge (beam − no beam) and absolute (field off, truth) keff now agree.** The old discrepancy came from the
  static model.
- **Against the current CDB**, the ring-dφ rms drops from 0.4–0.8 to about 0.3 mrad (beam − no beam and beam − field off).
- **Next:**
  - the track rDCA keff scan with the new static model;
  - the 83319 North outlier;
  - the remaining South dR shape.

# Backup
## B-C3. B_r × IFC with frame 0 (field-off reference with the same model)
**What is plotted.** For each comparison, B_r × {0.75, 0.80, 0.85} × IFC lowering per side, relative to the automatic IFC:
- the rms of the ring dφ;
- the shape dφ(R1) − dφ(R2,R3);
- then dφ and dR vs R per IFC lowering at B_r 0.8.

In [ ]:
for lab, runs, pre in n.F_CASES:
    print(f'CONFIG  {lab}: frame 0 / 0, B_r x IFC lowering, z -40, tilt 0.16 / 0.85' + (', alpha 1.6 at b 0.60' if pre == 'a1p6' else ''))
    show(n.f_map(lab, runs, pre, 'rms'), n.f_map(lab, runs, pre, 'shape'))
lab, runs, pre = n.F_CASES[2]
show(n.f_curves(lab, runs, pre, 5), n.f_curves(lab, runs, pre, 3))

## B-C4b. First-harmonic amplitude per TPC rotation / shift
RΔφ (top) and dR (bottom) per module ring, for no beam − field off and beam − field off.

In [ ]:
print('CONFIG  new static model, TPC rotation / x-y shift varied at z -40; field-off reference with the same model and geometry')
beam = ['79508', '79513', '79516', '79529']
show(n.tilt2_fig(['75073'], 'st_ns', 'no beam 75073 - field off', 4), n.tilt2_fig(beam, BEST, 'beam 795xx - field off', 4))
show(n.tilt2_fig(['83319'], 'st_ns', 'no beam 83319 - field off', 4))
show(n.tilt2_fig(['75073'], 'st_ns', 'no beam 75073 - field off', 3), n.tilt2_fig(beam, BEST, 'beam 795xx - field off', 3))
show(n.tilt2_fig(['83319'], 'st_ns', 'no beam 83319 - field off', 3))